# GPT-SoVITS v4 — Japanese Vocabulary App 全單字一鍵語音包

在 Colab 選「**執行階段 → 全部執行**」後，只需上傳一次參考音；筆記本會自動完成 GPT-SoVITS 安裝、模型下載、N5–N1 全部 **8,334** 個單字生成、完整性檢查、專案格式打包與下載。

- 詞彙 id、JLPT 等級和讀音已從目前專案的 `data/vocabulary/manifest.json` 內嵌，不需再上傳資料檔或 `stages.json`
- 不掛載 Google Drive；所有暫存與輸出都在本次 Colab 的 `/content`
- 輸出固定符合 `public/audio/voices/gpt-sovits-custom/<N級>/<word-id>.wav`
- 有效 WAV 會跳過，因此同一個 Colab 執行階段內可直接重跑批次格續跑
- 完成後下載的 zip 內含雙擊安裝器，會驗證 8,334 個檔案並自動註冊到目前專案

環境需求：Colab GPU 執行時間（T4 以上）。GPT-SoVITS 固定於 commit `d523079f`。

> 8,334 筆推理會花很長時間，免費 Colab 可能在完成前中斷；不使用持久儲存時，執行階段被回收後無法保留進度。請只使用你有權使用的參考音與聲線。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("CUDA 可用:", torch.cuda.is_available(), "| PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "請先到 [執行階段] → [變更執行階段類型] 選 GPU，再重開本筆記本"

In [ ]:
%cd /content
!rm -rf GPT-SoVITS
!git clone -q https://github.com/RVC-Boss/GPT-SoVITS.git
%cd GPT-SoVITS
!git checkout -q d523079fc05d9a8028d6085bffe4a2757c32abb6
!ls api.py config.py requirements.txt

## 安裝依賴（約 5–10 分鐘）

若安裝途中斷線，直接重跑中斷的格子即可；pip 會略過已完成項目。安裝拆成兩階段，降低 Colab RAM 壓力。

In [ ]:
import shutil
try:
    import psutil
    print("RAM:", round(psutil.virtual_memory().total / 2**30, 1), "GB | 可用:", round(psutil.virtual_memory().available / 2**30, 1), "GB")
except Exception:
    pass
print("磁碟可用:", round(shutil.disk_usage("/content").free / 2**30, 1), "GB")

In [ ]:
# 階段一：容易編譯失敗的套件先單獨安裝
!pip install --no-cache-dir --progress-bar off --disable-pip-version-check --upgrade huggingface_hub 2>&1 | tail -2
!pip install --no-cache-dir --progress-bar off --disable-pip-version-check opencc pyopenjtalk 2>&1 | tail -3
!pip install --no-cache-dir --progress-bar off --disable-pip-version-check jieba_fast python_mecab_ko 2>&1 | tail -3
print("階段一完成")

In [ ]:
# 階段二：其餘 GPT-SoVITS 依賴
%cd /content/GPT-SoVITS
!pip install --no-cache-dir --progress-bar off --disable-pip-version-check -r requirements.txt 2>&1 | tail -10
print("階段二完成")

In [ ]:
import importlib, sys
sys.path.insert(0, "/content/GPT-SoVITS/GPT_SoVITS")
sys.path.insert(0, "/content/GPT-SoVITS")

pip_modules = [
    "torch", "torchaudio", "librosa", "soundfile", "fastapi", "uvicorn",
    "transformers", "peft", "pytorch_lightning", "torchmetrics", "x_transformers",
    "pypinyin", "cn2an", "jieba", "jieba_fast", "opencc", "fast_langdetect",
    "split_lang", "wordsegment", "g2p_en", "sentencepiece", "funasr", "modelscope", "av",
]
local_modules = ["text.LangSegmenter", "text.cleaner"]
missing_pip, missing_local = [], []
for module_name in pip_modules:
    try:
        importlib.import_module(module_name)
    except Exception as error:
        missing_pip.append((module_name, str(error)[:80]))
for module_name in local_modules:
    try:
        importlib.import_module(module_name)
    except Exception as error:
        missing_local.append((module_name, str(error)[:80]))

if missing_pip:
    print("缺少 PyPI 套件：")
    for module_name, error in missing_pip:
        print(f"  - {module_name}: {error}")
if missing_local:
    print("GPT-SoVITS 內建模組載入失敗：")
    for module_name, error in missing_local:
        print(f"  - {module_name}: {error}")
assert not missing_pip and not missing_local, "依賴未完整，請重跑安裝格後再試"
print("依賴 OK")

In [ ]:
# 下載 v4 預訓練模型；重跑時會從 Hugging Face 快取續傳
import os
from huggingface_hub import snapshot_download

MODELS_DIR = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models"
os.makedirs(MODELS_DIR, exist_ok=True)
snapshot_download(
    repo_id="lj1995/GPT-SoVITS",
    allow_patterns=[
        "s1v3.ckpt",
        "gsv-v4-pretrained/*",
        "chinese-hubert-base/*",
        "chinese-roberta-wwm-ext-large/*",
    ],
    local_dir=MODELS_DIR,
)

# split_lang / fast_langdetect 第一次推理會在這裡下載語言辨識模型；套件不會自行建立父目錄
FAST_LANGDETECT_CACHE = f"{MODELS_DIR}/fast_langdetect"
os.makedirs(FAST_LANGDETECT_CACHE, exist_ok=True)
print("fast-langdetect 快取目錄：", FAST_LANGDETECT_CACHE)

required_models = [
    f"{MODELS_DIR}/s1v3.ckpt",
    f"{MODELS_DIR}/gsv-v4-pretrained/s2Gv4.pth",
    f"{MODELS_DIR}/gsv-v4-pretrained/vocoder.pth",
    f"{MODELS_DIR}/chinese-hubert-base/pytorch_model.bin",
    f"{MODELS_DIR}/chinese-roberta-wwm-ext-large/pytorch_model.bin",
    f"{MODELS_DIR}/chinese-roberta-wwm-ext-large/tokenizer.json",
    f"{MODELS_DIR}/chinese-roberta-wwm-ext-large/config.json",
]
missing_models = [path for path in required_models if not os.path.isfile(path)]
for path in required_models:
    print(("OK  " if os.path.isfile(path) else "缺檔 ") + path)
assert not missing_models, "模型檔案缺漏；請重跑本格"

## 唯一需要操作的步驟：上傳參考音

「全部執行」跑到這裡時，選一段 3–10 秒、單一說話者、無音樂且逐字稿已知的參考音。請把檔名直接命名為逐字稿，例如 `おはようございます.wav`；如果無法這樣命名，可先在下方處理格填 `PROMPT_TEXT_OVERRIDE`。

In [ ]:
%cd /content
from google.colab import files

print("請上傳一個參考音檔（wav / mp3 / m4a / flac / ogg）")
reference_upload = files.upload()
assert reference_upload, "沒有上傳參考音"
assert len(reference_upload) == 1, "一次只上傳一個參考音，請重跑本格"
REF_RAW = "/content/" + next(iter(reference_upload))
print("參考音來源：", REF_RAW)

In [ ]:
import os, re, subprocess
import librosa

# 檔名不是逐字稿時，請填入正確內容；例如：PROMPT_TEXT_OVERRIDE = "おはようございます"
PROMPT_TEXT_OVERRIDE = ""
PROMPT_LANG_OVERRIDE = ""  # 可填 ja / zh / en / ko / yue；留空自動判斷

REF_WAV = "/content/ref_32k.wav"
subprocess.run(
    ["ffmpeg", "-y", "-i", REF_RAW, "-ac", "1", "-ar", "32000", "-t", "10", "-loglevel", "error", REF_WAV],
    check=True,
)
duration = librosa.get_duration(path=REF_WAV)
assert 3 <= duration <= 10, f"參考音長度為 {duration:.1f} 秒；請改用 3–10 秒的音檔"

filename_transcript = os.path.splitext(os.path.basename(REF_RAW))[0].strip()
PROMPT_TEXT = PROMPT_TEXT_OVERRIDE.strip() or filename_transcript
assert PROMPT_TEXT, "請提供參考音逐字稿"

def detect_language(text):
    if re.search(r"[\u3040-\u30ff\u31f0-\u31ffー]", text):
        return "ja"
    if re.search(r"[\uac00-\ud7af]", text):
        return "ko"
    if re.fullmatch(r"[a-zA-Z0-9 .,!?'\"-]+", text):
        return "en"
    return "zh"

PROMPT_LANG = PROMPT_LANG_OVERRIDE.strip().lower() or detect_language(PROMPT_TEXT)
assert PROMPT_LANG in {"ja", "zh", "en", "ko", "yue"}, f"不支援的參考音語種：{PROMPT_LANG}"
print(f"參考音：{duration:.1f} 秒、32kHz 單聲道")
print("逐字稿：", PROMPT_TEXT)
print("語種：", PROMPT_LANG)

## 已內嵌目前專案的全部詞彙

本格會解壓並驗證 8,334 筆 N5–N1 詞彙資料，不需要上傳 `manifest.json` 或 `stages.json`。

In [ ]:
import base64, gzip, hashlib, json

LEVEL_ORDER = ["N5", "N4", "N3", "N2", "N1"]
SOURCE_VOCABULARY_SHA256 = "4dfd871216b76efc077d27f02b8f62c1995b3f6edb961f2357b8dcb6666b34f5"
EMBEDDED_PAYLOAD_SHA256 = "2c06d3c8c1f9d9c18de19561df9a798948a131e6361ef8c8b5e3a7add36abb9e"
EMBEDDED_DATA_B64 = """..."""

embedded_bytes = gzip.decompress(base64.b64decode(EMBEDDED_DATA_B64))
assert hashlib.sha256(embedded_bytes).hexdigest() == EMBEDDED_PAYLOAD_SHA256, "內嵌詞彙資料校驗失敗"
selected_words = json.loads(embedded_bytes.decode("utf-8"))
assert len(selected_words) == 8334, f"預期 8,334 詞，實際 {len(selected_words):,} 詞"
assert len({word["id"] for word in selected_words}) == 8334, "內嵌資料有重複 word id"
counts = {level: sum(word["jlpt"] == level for word in selected_words) for level in LEVEL_ORDER}
print("已載入全部詞彙：", counts, "總計", len(selected_words))

## 固定為全專案批次設定

下面已固定選取 N5–N1 全部 8,334 詞。預設優先念假名；資料中以 `/` 分隔的多讀音只念第一個版本，以符合目前一個 word id 對應一個 WAV 的架構。

In [ ]:
from pathlib import Path

VOICE_ID = "gpt-sovits-custom"
VOICE_NAME = "GPT-SoVITS 自訂音色"
LICENSE_NOTE = "Generated with GPT-SoVITS v4. Confirm reference-audio, model, and voice rights before redistribution."

TEXT_SOURCE = "kana"
FIRST_VARIANT_ONLY = True
ADD_END_PUNCTUATION = True
SPEED = 1.0
SAMPLE_STEPS = 8
TOP_K = 15
TOP_P = 0.6
TEMPERATURE = 0.6
MAX_RETRIES = 3
REQUEST_TIMEOUT = 300
OVERWRITE = False

WORK_ROOT = Path("/content/Japanese-TTS")
PACK_ROOT = WORK_ROOT / VOICE_ID
PACK_ROOT.mkdir(parents=True, exist_ok=True)

def clean_tts_text(word):
    value = str(word.get(TEXT_SOURCE) or word.get("kana") or word.get("display") or word.get("kanji") or "").strip()
    if FIRST_VARIANT_ONLY:
        value = value.split("/", 1)[0].strip()
    value = value.strip(" \t\r\n、。，．・")
    if not value:
        raise ValueError(f"{word.get('id')} 沒有可合成文字")
    return value

print(f"輸出：{PACK_ROOT}")
print(f"固定生成全部 {len(selected_words):,} 詞")
for word in selected_words[:10]:
    print(f"  {word['jlpt']}  {word['id']}  {clean_tts_text(word)}")

In [ ]:
# 啟動 GPT-SoVITS API（首次載入約 1–2 分鐘）
import os, subprocess, sys, time, requests

MODELS_DIR = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models"
FAST_LANGDETECT_CACHE = f"{MODELS_DIR}/fast_langdetect"
os.makedirs(FAST_LANGDETECT_CACHE, exist_ok=True)
required_models = [
    f"{MODELS_DIR}/s1v3.ckpt",
    f"{MODELS_DIR}/gsv-v4-pretrained/s2Gv4.pth",
    f"{MODELS_DIR}/gsv-v4-pretrained/vocoder.pth",
    f"{MODELS_DIR}/chinese-hubert-base/pytorch_model.bin",
    f"{MODELS_DIR}/chinese-roberta-wwm-ext-large/config.json",
]
missing_models = [path for path in required_models if not os.path.isfile(path)]
assert not missing_models, "模型檔案缺漏，請先重跑下載模型格：\n" + "\n".join(missing_models)

API = "http://127.0.0.1:9880/"
try:
    requests.get(API, timeout=2)
    api_ready = True
    print("沿用已啟動的 API")
except Exception:
    api_ready = False

if not api_ready:
    log_path = "/content/gpt_sovits_api.log"
    log_handle = open(log_path, "w")
    api_process = subprocess.Popen(
        [
            sys.executable, "api.py", "-a", "127.0.0.1", "-p", "9880",
            "-s", f"{MODELS_DIR}/gsv-v4-pretrained/s2Gv4.pth",
            "-g", f"{MODELS_DIR}/s1v3.ckpt",
            "-hb", f"{MODELS_DIR}/chinese-hubert-base",
            "-b", f"{MODELS_DIR}/chinese-roberta-wwm-ext-large",
        ],
        cwd="/content/GPT-SoVITS",
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )
    for _ in range(240):
        time.sleep(2)
        try:
            requests.get(API, timeout=2)
            api_ready = True
            break
        except Exception:
            if api_process.poll() is not None:
                break
    if not api_ready:
        log_handle.flush()
        print(open(log_path, encoding="utf-8", errors="replace").read()[-4000:])
        raise RuntimeError("API 啟動失敗；請查看上方 log")
print("API 就緒：", API)

## 自動產生一筆試聽

「全部執行」會先產生並顯示第一個單字的音訊，接著自動開始完整批次；這也同時驗證 API 回傳的是有效 WAV。

In [ ]:
import io, os, tempfile, time
from pathlib import Path
import requests, soundfile as sf
from requests.exceptions import RequestException
from IPython.display import Audio, display

def api_log_tail(limit=3000):
    log_file = Path("/content/gpt_sovits_api.log")
    if not log_file.exists():
        return "(找不到 API log)"
    return log_file.read_text(encoding="utf-8", errors="replace")[-limit:]

def validate_wav_payload(audio_bytes):
    if len(audio_bytes) < 44 or audio_bytes[:4] != b"RIFF" or audio_bytes[8:12] != b"WAVE":
        return False, "不是 RIFF/WAVE 資料"
    declared_size = int.from_bytes(audio_bytes[4:8], "little") + 8
    if declared_size > len(audio_bytes):
        return False, f"WAV 被截斷：宣告 {declared_size} bytes，只收到 {len(audio_bytes)} bytes"
    try:
        info = sf.info(io.BytesIO(audio_bytes))
        if info.frames <= 0 or info.samplerate <= 0 or info.channels < 1:
            return False, "WAV 沒有有效音訊 frame"
    except Exception as error:
        return False, f"WAV 解析失敗：{error}"
    return True, ""

def request_tts_bytes(text, transport_retries=3):
    request_text = text + "。" if ADD_END_PUNCTUATION and text[-1] not in "。！？!?" else text
    payload = {
        "refer_wav_path": REF_WAV,
        "prompt_text": PROMPT_TEXT,
        "prompt_language": PROMPT_LANG,
        "text": request_text,
        "text_language": "ja",
        "top_k": TOP_K,
        "top_p": TOP_P,
        "temperature": TEMPERATURE,
        "speed": SPEED,
        "sample_steps": SAMPLE_STEPS,
    }
    errors = []
    for attempt in range(1, transport_retries + 1):
        response = None
        chunks = []
        stream_error = None
        try:
            response = requests.post(
                API,
                json=payload,
                stream=True,
                headers={"Connection": "close"},
                timeout=(15, REQUEST_TIMEOUT),
            )
            try:
                for chunk in response.iter_content(chunk_size=64 * 1024):
                    if chunk:
                        chunks.append(chunk)
            except RequestException as error:
                stream_error = error

            received = b"".join(chunks)
            if response.status_code != 200:
                detail = received.decode("utf-8", errors="replace")[:500]
                raise RuntimeError(f"HTTP {response.status_code}：{detail}")

            valid, reason = validate_wav_payload(received)
            if valid:
                if stream_error:
                    print(f"串流結尾異常，但 WAV 已完整收到並通過驗證：{stream_error}")
                return received

            if stream_error:
                raise RuntimeError(f"{stream_error}; {reason}")
            raise RuntimeError(reason)
        except (RequestException, RuntimeError) as error:
            message = f"第 {attempt}/{transport_retries} 次傳輸失敗：{type(error).__name__}: {error}"
            errors.append(message)
            print(message)
            if attempt < transport_retries:
                time.sleep(min(2 ** attempt, 8))
        finally:
            if response is not None:
                response.close()

    raise RuntimeError(
        "GPT-SoVITS 音訊串流連續失敗。\n"
        + "\n".join(errors)
        + "\n\nAPI log 尾端：\n"
        + api_log_tail()
    )

preview_word = selected_words[0]
preview_text = clean_tts_text(preview_word)
preview_path = "/content/tts-preview.wav"
with open(preview_path, "wb") as handle:
    handle.write(request_tts_bytes(preview_text))
preview_info = sf.info(preview_path)
assert preview_info.frames > 0 and preview_info.samplerate > 0, "試聽 WAV 無效"
print(f"{preview_word['id']} | {preview_text} | {preview_info.samplerate}Hz | {preview_info.duration:.2f}s")
display(Audio(preview_path))

## 自動執行 N5–N1 全部生成

「全部執行」會直接進入本格，不需要再修改確認開關。每 10 筆顯示進度與預估剩餘時間；失敗項目會重試三次並寫入報告。同一執行階段重跑時會驗證並跳過既有 WAV。

In [ ]:
import json, time
from datetime import datetime, timezone
from pathlib import Path
import soundfile as sf

STATE_PATH = PACK_ROOT / "batch-state.json"
REPORT_PATH = PACK_ROOT / "generation-report.json"
FAILURES_PATH = PACK_ROOT / "failed.json"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    temp_path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temp_path.replace(path)

def valid_wav(path):
    try:
        info = sf.info(str(path))
        return info.frames > 0 and info.samplerate > 0 and info.channels >= 1
    except Exception:
        return False

state = {
    "voice_id": VOICE_ID,
    "started_at": utc_now(),
    "updated_at": utc_now(),
    "expected": len(selected_words),
    "completed": [],
    "failed": {},
}
if STATE_PATH.exists():
    try:
        old_state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
        if old_state.get("voice_id") == VOICE_ID:
            state["started_at"] = old_state.get("started_at", state["started_at"])
            state["failed"] = old_state.get("failed", {})
    except Exception as error:
        print("舊進度檔無法讀取，改從現有 WAV 掃描：", error)

completed_ids = []
failures = dict(state.get("failed", {}))
started = time.time()

try:
    for index, word in enumerate(selected_words, start=1):
        level, word_id = word["jlpt"], word["id"]
        text = clean_tts_text(word)
        output_path = PACK_ROOT / level / f"{word_id}.wav"
        output_path.parent.mkdir(parents=True, exist_ok=True)

        if not OVERWRITE and output_path.exists() and valid_wav(output_path):
            completed_ids.append(word_id)
            failures.pop(word_id, None)
        else:
            last_error = None
            for attempt in range(1, MAX_RETRIES + 1):
                temp_path = output_path.with_suffix(".wav.part")
                try:
                    audio_bytes = request_tts_bytes(text)
                    temp_path.write_bytes(audio_bytes)
                    info = sf.info(str(temp_path))
                    if info.frames <= 0 or info.samplerate <= 0:
                        raise RuntimeError("回傳 WAV 沒有有效音訊")
                    temp_path.replace(output_path)
                    completed_ids.append(word_id)
                    failures.pop(word_id, None)
                    last_error = None
                    break
                except Exception as error:
                    last_error = f"{type(error).__name__}: {error}"
                    if temp_path.exists():
                        temp_path.unlink()
                    print(f"失敗 {level}/{word_id} 第 {attempt} 次：{last_error}")
                    if attempt < MAX_RETRIES:
                        time.sleep(min(2 ** attempt, 10))
            if last_error:
                failures[word_id] = {
                    "level": level,
                    "text": text,
                    "error": last_error,
                    "updated_at": utc_now(),
                }

        if index == 1 or index % 10 == 0 or index == len(selected_words):
            elapsed = max(time.time() - started, 0.001)
            rate = index / elapsed
            eta_seconds = (len(selected_words) - index) / rate if rate > 0 else 0
            print(
                f"進度 {index:,}/{len(selected_words):,} | "
                f"成功 {len(completed_ids):,} | 失敗 {len(failures):,} | "
                f"ETA {eta_seconds / 3600:.1f} 小時"
            )
        if index % 25 == 0 or index == len(selected_words):
            state.update({"updated_at": utc_now(), "completed": completed_ids, "failed": failures})
            write_json_atomic(STATE_PATH, state)
except KeyboardInterrupt:
    print("已停止；同一個 Colab 執行階段可重跑本格續跑")
finally:
    state.update({"updated_at": utc_now(), "completed": completed_ids, "failed": failures})
    write_json_atomic(STATE_PATH, state)
    if failures:
        write_json_atomic(FAILURES_PATH, failures)
    elif FAILURES_PATH.exists():
        FAILURES_PATH.unlink()
    report = {
        "voice_id": VOICE_ID,
        "expected": len(selected_words),
        "completed": len(completed_ids),
        "failed": len(failures),
        "elapsed_seconds": round(time.time() - started, 1),
        "updated_at": utc_now(),
        "source_vocabulary_sha256": SOURCE_VOCABULARY_SHA256,
        "settings": {
            "text_source": TEXT_SOURCE,
            "first_variant_only": FIRST_VARIANT_ONLY,
            "speed": SPEED,
            "sample_steps": SAMPLE_STEPS,
            "top_k": TOP_K,
            "top_p": TOP_P,
            "temperature": TEMPERATURE,
        },
    }
    write_json_atomic(REPORT_PATH, report)
    print(json.dumps(report, ensure_ascii=False, indent=2))

assert len(completed_ids) == len(selected_words) and not failures, (
    "批次尚未完整；請保持此執行階段並重跑本格。"
    f" 成功 {len(completed_ids):,}/{len(selected_words):,}，失敗 {len(failures):,}。"
)

## 自動驗證、打包並下載

只有在 8,334 個 WAV 全部存在且有效時才會打包。zip 解壓後雙擊根目錄的 `INSTALL-VOICE-PACK.cmd`，安裝器會自動找到目前專案、再次核對 coverage，複製音檔並更新語音索引與 `src/audio.ts`。

In [ ]:
import json, shutil
from datetime import datetime, timezone
from pathlib import Path
import soundfile as sf
from google.colab import files

expected_by_level = {
    level: [word["id"] for word in selected_words if word["jlpt"] == level]
    for level in LEVEL_ORDER
}
coverage = {level: [] for level in LEVEL_ORDER}
sample_rates = set()
missing_files, invalid_files = [], []

for level in LEVEL_ORDER:
    for word_id in expected_by_level[level]:
        wav_path = PACK_ROOT / level / f"{word_id}.wav"
        if not wav_path.exists():
            missing_files.append(str(wav_path))
            continue
        try:
            info = sf.info(str(wav_path))
            if info.frames <= 0 or info.samplerate <= 0 or info.channels < 1:
                raise ValueError("empty or invalid wav")
            coverage[level].append(word_id)
            sample_rates.add(info.samplerate)
        except Exception as error:
            invalid_files.append({"file": str(wav_path), "error": str(error)})

assert not missing_files, f"缺少 {len(missing_files):,} 個 WAV；範例：{missing_files[:10]}"
assert not invalid_files, f"有 {len(invalid_files):,} 個無效 WAV；範例：{invalid_files[:10]}"
total_audio = sum(len(ids) for ids in coverage.values())
assert total_audio == 8334, f"預期 8,334 個 WAV，實際 {total_audio:,}"

voice_manifest = {
    "schema_version": 1,
    "id": VOICE_ID,
    "name": VOICE_NAME,
    "engine": "GPT-SoVITS",
    "engine_version": "v4-d523079f",
    "speaker": {"name": "zero-shot cloned voice", "reference_language": PROMPT_LANG},
    "language": "ja-JP",
    "format": "wav",
    "sample_rate": next(iter(sample_rates)) if len(sample_rates) == 1 else sorted(sample_rates),
    "license_note": LICENSE_NOTE,
    "source_vocabulary_sha256": SOURCE_VOCABULARY_SHA256,
    "coverage": coverage,
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
(PACK_ROOT / "manifest.json").write_text(
    json.dumps(voice_manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

export_root = Path("/content/japanese-tts-export")
if export_root.exists():
    shutil.rmtree(export_root)
export_pack = export_root / "public" / "audio" / "voices" / VOICE_ID
export_pack.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(
    PACK_ROOT,
    export_pack,
    ignore=shutil.ignore_patterns("batch-state.json", "failed.json", "*.part", "*.tmp"),
)

installer_ps1 = '[CmdletBinding()]\nparam(\n    [string]$ProjectRoot = "",\n    [string]$PackageRoot = "",\n    [switch]$SkipCoverageCheck\n)\n\n$ErrorActionPreference = "Stop"\n$VoiceId = "gpt-sovits-custom"\n$InstallerRoot = Split-Path -Parent $MyInvocation.MyCommand.Path\n$Utf8NoBom = New-Object System.Text.UTF8Encoding($false)\n\nfunction Test-ProjectRoot([string]$Path) {\n    if ([string]::IsNullOrWhiteSpace($Path)) { return $false }\n    return (\n        (Test-Path -LiteralPath (Join-Path $Path "package.json")) -and\n        (Test-Path -LiteralPath (Join-Path $Path "src\\audio.ts")) -and\n        (Test-Path -LiteralPath (Join-Path $Path "data\\vocabulary\\manifest.json")) -and\n        (Test-Path -LiteralPath (Join-Path $Path "public\\audio\\voices\\index.json"))\n    )\n}\n\nfunction Resolve-ExistingPath([string]$Path) {\n    return (Resolve-Path -LiteralPath $Path).Path.TrimEnd(\'\\\')\n}\n\nif ([string]::IsNullOrWhiteSpace($PackageRoot)) {\n    $PackageCandidates = @(\n        (Join-Path $InstallerRoot "public\\audio\\voices\\$VoiceId"),\n        (Join-Path (Split-Path -Parent $InstallerRoot) "public\\audio\\voices\\$VoiceId")\n    )\n    $PackageRoot = $PackageCandidates | Where-Object {\n        Test-Path -LiteralPath (Join-Path $_ "manifest.json")\n    } | Select-Object -First 1\n}\n\nif ([string]::IsNullOrWhiteSpace($PackageRoot) -or -not (Test-Path -LiteralPath (Join-Path $PackageRoot "manifest.json"))) {\n    throw "Voice package not found. Extract the downloaded zip first, or pass -PackageRoot."\n}\n$PackageRoot = Resolve-ExistingPath $PackageRoot\n\nif ([string]::IsNullOrWhiteSpace($ProjectRoot)) {\n    $ProjectCandidates = New-Object System.Collections.Generic.List[string]\n    $ProjectCandidates.Add((Get-Location).Path)\n    $Cursor = $InstallerRoot\n    for ($Index = 0; $Index -lt 6; $Index++) {\n        $ProjectCandidates.Add($Cursor)\n        $Parent = Split-Path -Parent $Cursor\n        if ([string]::IsNullOrWhiteSpace($Parent) -or $Parent -eq $Cursor) { break }\n        $Cursor = $Parent\n    }\n    $ProjectCandidates.Add("D:\\codex projects\\Japanese")\n    $ProjectRoot = $ProjectCandidates | Where-Object { Test-ProjectRoot $_ } | Select-Object -First 1\n}\n\nif (-not (Test-ProjectRoot $ProjectRoot)) {\n    throw "Japanese project not found. Run with -ProjectRoot followed by the project directory."\n}\n$ProjectRoot = Resolve-ExistingPath $ProjectRoot\n\n$PackManifestPath = Join-Path $PackageRoot "manifest.json"\n$PackManifest = Get-Content -LiteralPath $PackManifestPath -Raw -Encoding UTF8 | ConvertFrom-Json\nif ($PackManifest.id -ne $VoiceId) {\n    throw "Unexpected voice id in package manifest: $($PackManifest.id)"\n}\n$VoiceName = [string]$PackManifest.name\nif ([string]::IsNullOrWhiteSpace($VoiceName)) { $VoiceName = "GPT-SoVITS custom voice" }\n\nif (-not $SkipCoverageCheck) {\n    $VocabularyPath = Join-Path $ProjectRoot "data\\vocabulary\\manifest.json"\n    $Vocabulary = Get-Content -LiteralPath $VocabularyPath -Raw -Encoding UTF8 | ConvertFrom-Json\n    $Missing = New-Object System.Collections.Generic.List[string]\n    foreach ($Level in @("N5", "N4", "N3", "N2", "N1")) {\n        $Covered = @{}\n        foreach ($WordId in @($PackManifest.coverage.$Level)) {\n            $Covered[[string]$WordId] = $true\n        }\n        foreach ($Word in @($Vocabulary.levels.$Level.items)) {\n            $WordId = [string]$Word.id\n            $SourceWav = Join-Path $PackageRoot "$Level\\$WordId.wav"\n            if (-not $Covered.ContainsKey($WordId) -or -not (Test-Path -LiteralPath $SourceWav)) {\n                $Missing.Add("$Level/$WordId.wav")\n                if ($Missing.Count -ge 20) { break }\n            }\n        }\n        if ($Missing.Count -ge 20) { break }\n    }\n    if ($Missing.Count -gt 0) {\n        throw "Voice package is incomplete. First missing files: $($Missing -join \', \')"\n    }\n}\n\n$DestinationRoot = Join-Path $ProjectRoot "public\\audio\\voices\\$VoiceId"\n$DestinationParent = Split-Path -Parent $DestinationRoot\nNew-Item -ItemType Directory -Path $DestinationParent -Force | Out-Null\n\n$SourceResolved = Resolve-ExistingPath $PackageRoot\n$DestinationResolved = $null\nif (Test-Path -LiteralPath $DestinationRoot) {\n    $DestinationResolved = Resolve-ExistingPath $DestinationRoot\n}\nif ($SourceResolved -ne $DestinationResolved) {\n    New-Item -ItemType Directory -Path $DestinationRoot -Force | Out-Null\n    Get-ChildItem -LiteralPath $PackageRoot -Force | Copy-Item -Destination $DestinationRoot -Recurse -Force\n}\n\n$VoiceIndexPath = Join-Path $ProjectRoot "public\\audio\\voices\\index.json"\n$VoiceIndex = Get-Content -LiteralPath $VoiceIndexPath -Raw -Encoding UTF8 | ConvertFrom-Json\n$NewVoice = [pscustomobject]@{\n    id = $VoiceId\n    name = $VoiceName\n    type = "audio-pack"\n    manifest = "/audio/voices/$VoiceId/manifest.json"\n}\n$VoiceIndex.voices = @($VoiceIndex.voices | Where-Object { $_.id -ne $VoiceId }) + @($NewVoice)\n$VoiceIndexJson = $VoiceIndex | ConvertTo-Json -Depth 20\n[System.IO.File]::WriteAllText($VoiceIndexPath, $VoiceIndexJson + "`n", $Utf8NoBom)\n\n$AudioSourcePath = Join-Path $ProjectRoot "src\\audio.ts"\n$AudioSource = [System.IO.File]::ReadAllText($AudioSourcePath, [System.Text.Encoding]::UTF8)\n$NewLine = if ($AudioSource.Contains("`r`n")) { "`r`n" } else { "`n" }\n$TypePattern = \'(?m)^export type VoiceId = (?<types>[^;]+);\'\n$TypeMatch = [regex]::Match($AudioSource, $TypePattern)\nif (-not $TypeMatch.Success) {\n    throw "Could not find VoiceId in src/audio.ts"\n}\nif ($TypeMatch.Groups[\'types\'].Value -notmatch [regex]::Escape(\'"\' + $VoiceId + \'"\')) {\n    $Replacement = $TypeMatch.Value.TrimEnd(\';\') + \' | "\' + $VoiceId + \'";\'\n    $AudioSource = $AudioSource.Remove($TypeMatch.Index, $TypeMatch.Length).Insert($TypeMatch.Index, $Replacement)\n}\n\n$OptionsStart = $AudioSource.IndexOf("export const voiceOptions: VoiceOption[] = [")\nif ($OptionsStart -lt 0) {\n    throw "Could not find voiceOptions in src/audio.ts"\n}\n$OptionsEnd = $AudioSource.IndexOf($NewLine + "];", $OptionsStart)\nif ($OptionsEnd -lt 0) {\n    throw "Could not find the end of voiceOptions in src/audio.ts"\n}\n$OptionsBlock = $AudioSource.Substring($OptionsStart, $OptionsEnd - $OptionsStart)\nif ($OptionsBlock -notmatch (\'id:\\s*"\' + [regex]::Escape($VoiceId) + \'"\')) {\n    $TsVoiceName = $VoiceName.Replace(\'\\\', \'\\\\\').Replace(\'"\', \'\\"\')\n    $Entry = $NewLine + "  {" +\n        $NewLine + "    id: `"$VoiceId`"," +\n        $NewLine + "    name: `"$TsVoiceName`"," +\n        $NewLine + "    type: `"audio-pack`"," +\n        $NewLine + "    format: `"wav`"," +\n        $NewLine + "  },"\n    $AudioSource = $AudioSource.Insert($OptionsEnd, $Entry)\n}\n[System.IO.File]::WriteAllText($AudioSourcePath, $AudioSource, $Utf8NoBom)\n\nWrite-Host "Installed $VoiceId into: $ProjectRoot"\nWrite-Host "Audio folder: $DestinationRoot"\nWrite-Host "Updated: public/audio/voices/index.json"\nWrite-Host "Updated: src/audio.ts"\n\n'
installer_cmd = '@echo off\nsetlocal\npowershell.exe -NoProfile -ExecutionPolicy Bypass -File "%~dp0install-gpt-sovits-voice-pack.ps1"\nif errorlevel 1 (\n  echo.\n  echo Installation failed. See the message above.\n  pause\n  exit /b 1\n)\necho.\necho Voice pack installation completed.\npause\n'
(export_root / "install-gpt-sovits-voice-pack.ps1").write_text(installer_ps1, encoding="ascii")
(export_root / "INSTALL-VOICE-PACK.cmd").write_text(installer_cmd, encoding="ascii")
(export_root / "README-INSTALL.txt").write_text(
    "1. Extract this zip.\n"
    "2. Double-click INSTALL-VOICE-PACK.cmd.\n"
    "3. The installer validates all 8,334 files and updates the Japanese project automatically.\n",
    encoding="ascii",
)

zip_base = f"/content/{VOICE_ID}-complete-voice-pack"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=export_root)
print(f"已完整打包 {total_audio:,} 個 WAV：{zip_path}")
print("coverage：", {level: len(ids) for level, ids in coverage.items()})
files.download(zip_path)

## 使用方式與限制

1. 在 Colab 選 GPU 後按「執行階段 → 全部執行」。
2. 出現上傳視窗時，選擇以逐字稿命名的參考音。
3. 等待 8,334 筆全部生成；完成時瀏覽器會下載 zip。
4. 解壓 zip，雙擊 `INSTALL-VOICE-PACK.cmd`。

安裝器預設會自動辨識目前工作目錄、解壓位置的上層，以及 `D:\codex projects\Japanese`。若專案搬家，可在 PowerShell 執行：

```powershell
.\install-gpt-sovits-voice-pack.ps1 -ProjectRoot "新的專案路徑"
```

- API log：`/content/gpt_sovits_api.log`
- 目前使用 v4 的快速設定 `SAMPLE_STEPS = 8`；若更重視品質，可改成 `32`
- 同一執行階段中斷生成：直接重跑「自動執行 N5–N1 全部生成」格
- Colab 執行階段被回收：因依照要求未掛載 Google Drive，已生成檔案無法復原

GPT-SoVITS：[RVC-Boss/GPT-SoVITS](https://github.com/RVC-Boss/GPT-SoVITS)